In [1]:
import pandas as pd
from Bio.SeqUtils.ProtParam import ProteinAnalysis

In [2]:
STANDARD = set("ACDEFGHIKLMNPQRSTVWY")
df = pd.read_csv("raw_peptides.csv")
print("Loaded:", len(df))

df = df.drop_duplicates(subset="sequence")
df = df[df["sequence"].apply(lambda s: set(s.upper()).issubset(STANDARD))]
df = df[df["sequence"].str.len().between(8, 20)]
print("After cleaning:", len(df))

Loaded: 3913
After cleaning: 3913


In [3]:
def get_properties(seq):
    pa = ProteinAnalysis(seq.strip().upper())
    aa = pa.amino_acids_percent
    return pd.Series({
        "molecular_weight"  : round(pa.molecular_weight(), 3),
        "instability_index" : round(pa.instability_index(), 3),
        "isoelectric_point" : round(pa.isoelectric_point(), 3),
        "aromaticity"       : round(pa.aromaticity(), 3),
        "helix_fraction"    : round(pa.secondary_structure_fraction()[0], 3),
        "turn_fraction"     : round(pa.secondary_structure_fraction()[1], 3),
        "sheet_fraction"    : round(pa.secondary_structure_fraction()[2], 3),
        "pct_hydrophobic"   : round(sum(aa.get(a,0) for a in "AILMFWYV"), 3),
        "pct_positive"      : round(sum(aa.get(a,0) for a in "KRH"), 3),
        "pct_negative"      : round(sum(aa.get(a,0) for a in "DE"), 3),
        "pct_polar"         : round(sum(aa.get(a,0) for a in "STNQ"), 3),
    })

In [4]:
props = df["sequence"].apply(get_properties)
df = pd.concat([df.reset_index(drop=True), props], axis=1)
df = df.dropna(subset=["molecular_weight"])
df.to_csv("training_dataset.csv", index=False)
print("Saved! Total peptides:", len(df))

Saved! Total peptides: 3913
